# Prosty model językowy (char-level LLM) – wprowadzenie

W tym notebooku krok po kroku zbudujesz i wytrenujesz własny prosty model językowy oparty na sieci neuronowej z warstwą embedding i warstwami liniowymi (MLP).

Celem jest zrozumienie, jak:
- przygotować dane tekstowe do uczenia maszynowego,
- zakodować tekst na liczby (tokenizacja na znaki),
- zbudować i wytrenować własny, prosty model językowy,
- monitorować postępy treningu i generować tekst,
- testować i eksperymentować z własnym modelem.

Projekt nie ma na celu stworzenia w pełni funkcjonalnego modelu LLM na miarę GPT czy Claude, lecz pozwala poznać podstawowe mechanizmy działania takich sieci neuronowych oraz cały proces przygotowania, treningu i testowania własnego modelu.

# Przygotowanie danych do treningu LLM

W tym notebooku przygotowujemy dane tekstowe do treningu prostego modelu językowego. Kroki:

1. **Wczytanie tekstu**  
   Wczytujemy plik tekstowy z przykładowym korpusem.

2. **Czyszczenie tekstu**  
   Zamieniamy tekst na małe litery, usuwamy znaki nowej linii oraz znaki specjalne, pozostawiając tylko litery, cyfry i spacje.

3. **Tokenizacja na znaki**  
   Tworzymy słownik znaków (każdy unikalny znak otrzymuje swój numer) i zamieniamy tekst na listę indeksów (tokenów).

Dzięki temu uzyskujemy dane wejściowe w postaci sekwencji liczb, które można wykorzystać do treningu modelu językowego.

In [ ]:
# 1. Wczytaj tekst
with open('../data/raw/sample.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 2. Czyszczenie tekstu
text = text.lower()
text = text.replace('\n', ' ').replace('\r', '')
text = ''.join(c for c in text if c.isalnum() or c.isspace())

# 3. Tokenizacja na znaki
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encoded = [stoi[c] for c in text]

print(f"Liczba unikalnych znaków: {len(chars)}")
print(f"Przykładowe zakodowane dane: {encoded[:20]}")

# Budowa prostego modelu językowego (char-level LLM)

W tym kroku stworzymy prosty model językowy, który przewiduje kolejny znak na podstawie poprzednich znaków w sekwencji.  
To podstawowa wersja modelu typu "character-level language model" – podobna do uproszczonego nano-GPT, ale bez transformera.

## Założenia:
- Model przyjmuje sekwencję znaków (zakodowanych jako liczby) i uczy się przewidywać następny znak.
- Używamy architektury z warstwą embedding i kilkoma warstwami liniowymi (MLP).
- Model będzie trenowany na przygotowanych wcześniej danych tekstowych.

## Kluczowe elementy modelu:
1. **Warstwa embedding** – zamienia indeksy znaków na wektory liczbowe.
2. **Warstwa liniowa (MLP)** – przetwarza uśrednione wektory sekwencji.
3. **Warstwa wyjściowa** – przewiduje prawdopodobieństwo wystąpienia każdego znaku na kolejnej pozycji.

## Cel:
Zrozumieć, jak zbudować i zaimplementować od podstaw prosty model językowy, który można trenować na własnych danych tekstowych.

In [ ]:
import torch
import torch.nn as nn

class CharLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        # Warstwa embedding: zamienia indeksy znaków na wektory
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Prosta warstwa liniowa (możesz rozbudować o kolejne warstwy lub dodać transformer)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        # x: [batch_size, seq_length]
        x = self.embedding(x)           # [batch_size, seq_length, embedding_dim]
        x = x.mean(dim=1)               # Uśredniamy po sekwencji (proste podejście)
        x = self.fc1(x)
        x = self.relu(x)
        logits = self.fc2(x)            # [batch_size, vocab_size]
        return logits

# Przykład użycia:
vocab_size = len(stoi)      # liczba unikalnych znaków
embedding_dim = 32          # wymiar wektora osadzenia
hidden_dim = 64             # wymiar ukrytej warstwy

model = CharLanguageModel(vocab_size, embedding_dim, hidden_dim)

# Wydrukuj model
print(model)

# Wyświetlenie liczby parametrów modelu
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Liczba parametrów modelu: {count_parameters(model)}")

# Przygotowanie danych do treningu modelu językowego

W tym kroku przygotujemy dane wejściowe i wyjściowe do treningu modelu.  
Podzielimy zakodowany tekst na sekwencje o stałej długości (`seq_length`).  
Dla każdej sekwencji wejściowej (ciąg znaków) przygotujemy odpowiadający jej znak wyjściowy (kolejny znak po sekwencji).  
Dzięki temu model będzie uczył się przewidywać następny znak na podstawie poprzednich.

**Kroki:**
1. Ustalamy długość sekwencji (`seq_length`).
2. Tworzymy listy sekwencji wejściowych (`X`) i odpowiadających im znaków wyjściowych (`y`).
3. Zamieniamy je na tensory PyTorch do treningu.

In [ ]:
import torch

# Załaduj zakodowane dane (np. z poprzedniego notebooka)

seq_length = 32  # długość sekwencji wejściowej
X = []
y = []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i+seq_length])
    y.append(encoded[i+seq_length])

# Konwersja do tensorów
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f"Kształt X: {X.shape}")  # [liczba próbek, seq_length]
print(f"Kształt y: {y.shape}")  # [liczba próbek]

# Implementacja pętli treningowej

W tym kroku zaimplementujemy pętlę treningową dla naszego modelu językowego.  
Zdefiniujemy funkcję straty (`CrossEntropyLoss`) oraz optymalizator (`Adam`).  
W każdej epoce model będzie wykonywał prognozę, obliczał stratę, wykonywał propagację wsteczną i aktualizował swoje parametry.  
Będziemy monitorować wartość straty, aby obserwować postępy uczenia.

**Kroki:**
1. Zdefiniuj funkcję straty i optymalizator.
2. Wykonaj pętlę treningową przez określoną liczbę epok.
3. W każdej epoce oblicz stratę i zaktualizuj parametry modelu.

In [ ]:
# ...istniejący kod...

import torch.nn as nn
import torch.optim as optim

# Parametry treningu
epochs = 10
batch_size = 32
learning_rate = 1e-3

# Funkcja straty i optymalizator
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Trening
for epoch in range(epochs):
    # Losowa permutacja indeksów próbek (shuffle danych w każdej epoce)
    permutation = torch.randperm(X.size(0))
    epoch_loss = 0  # Suma strat dla danej epoki

    # Przechodzimy po danych partiami (batchami)
    for i in range(0, X.size(0), batch_size):
        # Wybieramy indeksy do aktualnej partii
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X[indices], y[indices]  # Batch wejściowy i wyjściowy

        optimizer.zero_grad()               # Zerujemy gradienty z poprzedniej iteracji
        outputs = model(batch_x)            # Przepuszczamy batch przez model (prognozy)
        loss = criterion(outputs, batch_y)  # Obliczamy stratę dla batcha
        loss.backward()                     # Propagacja wsteczna – obliczenie gradientów
        optimizer.step()                    # Aktualizacja parametrów modelu
        epoch_loss += loss.item()           # Dodajemy stratę do sumy strat w tej epoce

    avg_loss = epoch_loss / (X.size(0) // batch_size)
    print(f"Epoka {epoch+1}/{epochs}, strata: {avg_loss:.4f}")

# Monitorowanie postępów i generowanie tekstu

W tym kroku będziemy monitorować postępy treningu, generując przykładowy tekst z modelu po każdej epoce.  
Dzięki temu zobaczymy, jak model uczy się przewidywać kolejne znaki i czy generowany tekst zaczyna przypominać dane treningowe.

**Kroki:**
1. Po każdej epoce treningu wygeneruj przykładowy tekst, zaczynając od losowego znaku lub krótkiej sekwencji.
2. W każdym kroku generowania model przewiduje kolejny znak, który jest dodawany do sekwencji wejściowej.
3. Powtarzaj, aż uzyskasz żądaną długość wygenerowanego tekstu.

In [ ]:
import random

def generate_text(model, stoi, itos, start_text='', length=100):
    model.eval()  # Przełącz model w tryb ewaluacji (wyłącza dropout, batchnorm itp.)
    if not start_text:
        # Jeśli nie podano tekstu startowego, wybierz losowy znak z alfabetu
        start_text = random.choice(list(stoi.keys()))
    input_seq = [stoi[c] for c in start_text]                   # Zamień znaki startowe na indeksy
    generated = start_text                                      # Zainicjuj wygenerowany tekst tekstem startowym

    for _ in range(length):
        # Przygotuj tensor wejściowy: ostatnie seq_length znaków (lub mniej na początku)
        x = torch.tensor([input_seq[-seq_length:]], dtype=torch.long)   # [1, seq_length]
        with torch.no_grad():                                           # Nie śledzimy gradientów podczas generowania
            logits = model(x)                                           # Przepuść sekwencję przez model, otrzymaj logity
            probs = torch.softmax(logits, dim=-1)                       # Zamień logity na prawdopodobieństwa
            next_id = torch.multinomial(probs, num_samples=1).item()    # Wylosuj kolejny znak wg rozkładu
        generated += itos[next_id]                                      # Dodaj wygenerowany znak do tekstu
        input_seq.append(next_id)                                       # Dodaj indeks wygenerowanego znaku do sekwencji wejściowej
    model.train()                                                       # Przełącz model z powrotem w tryb treningowy
    return generated

# Trening modelu z generowaniem tekstu po każdej epoce

W tym kroku połączymy pętlę treningową z funkcją generowania tekstu.  
Po każdej epoce treningu model wygeneruje przykładowy tekst, co pozwoli na bieżąco obserwować, jak poprawia się jego zdolność do przewidywania kolejnych znaków.

In [ ]:
# Parametry treningu
epochs = 150
learning_rate = 1e-4

for epoch in range(epochs):
    permutation = torch.randperm(X.size(0))
    epoch_loss = 0

    for i in range(0, X.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X[indices], y[indices]

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / (X.size(0) // batch_size)
    print(f"Epoka {epoch+1}/{epochs}, strata: {avg_loss:.4f}")

    # Generowanie przykładowego tekstu po każdej epoce
    print("Przykładowy tekst generowany przez model:")
    print(generate_text(model, stoi, itos, start_text='według oryginalnego znaczenia', length=32)) # Według oryginalnego znaczenia terminu, robot jest kontrolowany przez sztuczną inteligencję
    print('-' * 80)

# Eksperymenty

W tej części możesz samodzielnie eksperymentować z różnymi parametrami modelu i procesu uczenia, aby sprawdzić, jak wpływają one na jakość generowanego tekstu.  
Oto przykładowe elementy, które warto zmieniać i obserwować:

- **Wielkość warstwy embedding** (`embedding_dim`) – większa liczba może pozwolić modelowi lepiej reprezentować znaki.
- **Liczba i rozmiar warstw liniowych** (`hidden_dim`, liczba warstw) – więcej lub większe warstwy mogą zwiększyć możliwości modelu, ale też ryzyko przeuczenia.
- **Długość sekwencji wejściowej** (`seq_length`) – dłuższe sekwencje pozwalają modelowi uczyć się dłuższych zależności w tekście.
- **Liczba epok treningu** (`epochs`) – więcej epok to dłuższy trening, ale nie zawsze lepsze rezultaty.
- **Rozmiar batcha** (`batch_size`) – wpływa na stabilność i szybkość uczenia.
- **Współczynnik uczenia** (`learning_rate`) – zbyt duży może destabilizować uczenie, zbyt mały spowolni postępy.
- **Tekst treningowy** – spróbuj użyć innego korpusu lub własnych danych.

Zachęcam do testowania różnych ustawień i obserwowania, jak zmienia się jakość generowanego tekstu.

# Prosty interfejs chatbota z modelem językowym w Gradio

Dzięki bibliotece [Gradio](https://gradio.app/) możesz szybko uruchomić prosty webowy interfejs do rozmowy z własnym modelem językowym.  
Wpisz tekst, a model wygeneruje odpowiedź bezpośrednio w przeglądarce.

In [ ]:
import gradio as gr

def gradio_chat(user_input):
    # Uzupełnij lub przytnij tekst do seq_length
    if len(user_input) < seq_length:
        prompt = ' ' * (seq_length - len(user_input)) + user_input
    else:
        prompt = user_input[-seq_length:]
    response = generate_text(model, stoi, itos, start_text=prompt, length=200)
    # Zwróć tylko wygenerowaną odpowiedź (bez promptu)
    return response[len(prompt):]

iface = gr.Interface(
    fn=gradio_chat,
    inputs=gr.Textbox(lines=2, label="Twój tekst"),
    outputs=gr.Textbox(lines=10, label="Odpowiedź modelu"),
    title="Prosty Chatbot LLM",
    description="Porozmawiaj z własnym modelem językowym!"
)

iface.launch()

# Zapis wytrenowanego modelu do pliku

Po zakończonym treningu warto zapisać wytrenowany model do pliku, aby móc go później łatwo wczytać i używać bez ponownego trenowania.  
W PyTorch do tego celu używa się funkcji `torch.save()`.

**Jak to zrobić:**
- Zapisz stan modelu (`state_dict`) do pliku `.pt` lub `.pth`.
- W razie potrzeby zapisz też słowniki `stoi` i `itos`.

In [ ]:
# ...po zakończeniu treningu...

import torch

# Zapisz model
torch.save(model.state_dict(), "../model.pth")

# Zapisz słowniki (opcjonalnie)
import pickle
with open("../stoi.pkl", "wb") as f:
    pickle.dump(stoi, f)
with open("../itos.pkl", "wb") as f:
    pickle.dump(itos, f)

print("Model i słowniki zostały zapisane do plików.")